# Cell 1 – Imports and paths:

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

CATALOG = "workspace"

ROOT = f"/Volumes/{CATALOG}/landing/staging/noaa"
NEW_DIR = f"{ROOT}/new"
DONE_DIR = f"{ROOT}/processed"
STATE_DIR = f"{ROOT}/_state"

BRONZE_TABLE = f"{CATALOG}.bronze.noaa_space_weather_raw"

spark.conf.set("spark.sql.session.timeZone", "UTC")

dbutils.fs.mkdirs(NEW_DIR)
dbutils.fs.mkdirs(DONE_DIR)
dbutils.fs.mkdirs(STATE_DIR)

print("NOAA staging:", ROOT)
print("NOAA Bronze table:", BRONZE_TABLE)

NOAA staging: /Volumes/workspace/landing/staging/noaa
NOAA Bronze table: workspace.bronze.noaa_space_weather_raw


# Cell 2 – Explicit input schema:

In [0]:
NOAA_INPUT_SCHEMA = StructType([
    StructField("observation_timestamp", StringType(), True),
    StructField("source", StringType(), True),
    StructField("raw_payload", StringType(), True),
    StructField("source_file", StringType(), True),
])

print(NOAA_INPUT_SCHEMA.simpleString())

struct<observation_timestamp:string,source:string,raw_payload:string,source_file:string>


# Cell 3 – Create the Bronze Delta table:

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
        observation_timestamp STRING,
        source                STRING,
        raw_payload           STRING,
        source_file           STRING
    )
    USING DELTA
""")

display(spark.sql(f"DESCRIBE TABLE {BRONZE_TABLE}"))

col_name,data_type,comment
observation_timestamp,string,null
source,string,null
raw_payload,string,null
source_file,string,null


# Cell 4 – Read, impose schema, and merge batches:

In [0]:
# Read the staged filenames. The collector writes JSON Lines files here.
jsonl_files = sorted(
    [
        f for f in dbutils.fs.ls(NEW_DIR)
        if f.name.endswith(".jsonl")
    ],
    key=lambda f: f.name,
)

print("NOAA batches waiting:", len(jsonl_files))

# Keep a record of batches already loaded. This prevents reloading a
# previously processed file if it is accidentally copied back to new/.
already_loaded = {
    row["source_file"]
    for row in (
        spark.table(BRONZE_TABLE)
        .select("source_file")
        .where(F.col("source_file").isNotNull())
        .distinct()
        .collect()
    )
}

for staged_file in jsonl_files:
    filename = staged_file.name

    if filename in already_loaded:
        print("Batch already loaded; moving it to processed:", filename)
        dbutils.fs.mv(
            staged_file.path,
            f"{DONE_DIR}/{filename}"
        )
        continue

    # Schema imposition is intentional: do not remove .schema(...).
    batch = (
        spark.read
        .schema(NOAA_INPUT_SCHEMA)
        .json(staged_file.path)
    )

    # Reject empty batches and missing required fields.
    if batch.limit(1).count() == 0:
        raise ValueError(
            f"{filename} is empty. It remains in new/ for investigation."
        )

    invalid = batch.filter(
        F.col("observation_timestamp").isNull()
        | (F.trim(F.col("observation_timestamp")) == "")
        | F.col("source").isNull()
        | (F.trim(F.col("source")) == "")
        | F.col("raw_payload").isNull()
        | F.col("source_file").isNull()
    )

    if invalid.limit(1).count() > 0:
        display(invalid.limit(20))
        raise ValueError(
            f"{filename} has invalid required fields. "
            "The batch remains in new/."
        )

    # Only the expected NOAA source labels are accepted.
    unexpected_sources = (
        batch
        .filter(
            ~F.col("source").isin(
                "noaa_swpc_kp",
                "noaa_swpc_f107",
            )
        )
        .select("source")
        .distinct()
    )

    if unexpected_sources.limit(1).count() > 0:
        display(unexpected_sources)
        raise ValueError(
            f"{filename} contains unexpected source labels."
        )

    # The merge key follows the existing Phase 1 Bronze contract.
    # If NOAA's source semantics later require multiple records per
    # timestamp, revisit this key before downstream modeling.
    batch.createOrReplaceTempView("noaa_incoming_batch")

    spark.sql(f"""
        MERGE INTO {BRONZE_TABLE} AS target
        USING (
            SELECT
                observation_timestamp,
                source,
                raw_payload,
                source_file
            FROM noaa_incoming_batch
            QUALIFY ROW_NUMBER() OVER (
                PARTITION BY source, observation_timestamp
                ORDER BY source_file DESC
            ) = 1
        ) AS incoming
        ON target.source = incoming.source
           AND target.observation_timestamp =
               incoming.observation_timestamp

        WHEN MATCHED THEN UPDATE SET
            target.raw_payload = incoming.raw_payload,
            target.source_file = incoming.source_file

        WHEN NOT MATCHED THEN INSERT (
            observation_timestamp,
            source,
            raw_payload,
            source_file
        )
        VALUES (
            incoming.observation_timestamp,
            incoming.source,
            incoming.raw_payload,
            incoming.source_file
        )
    """)

    print(
        f"Merged: {filename} | rows: {batch.count()}"
    )

    # Move only after validation and merge succeed.
    dbutils.fs.mv(
        staged_file.path,
        f"{DONE_DIR}/{filename}"
    )

    already_loaded.add(filename)

print("NOAA Bronze batch processing complete.")

NOAA batches waiting: 1
Merged: noaa_full_20261009T161409Z.jsonl | rows: 1822
NOAA Bronze batch processing complete.


# Cell 5 – Health check:

In [0]:
display(spark.sql(f"""
    SELECT
        source,
        COUNT(*) AS observations,
        COUNT(DISTINCT observation_timestamp) AS distinct_timestamps,
        MIN(try_to_timestamp(observation_timestamp)) AS first_timestamp_utc,
        MAX(try_to_timestamp(observation_timestamp)) AS last_timestamp_utc,
        SUM(
            CASE WHEN raw_payload IS NULL THEN 1 ELSE 0 END
        ) AS null_payloads
    FROM {BRONZE_TABLE}
    GROUP BY source
    ORDER BY source
"""))

source,observations,distinct_timestamps,first_timestamp_utc,last_timestamp_utc,null_payloads
noaa_swpc_f107,209,209,2026-03-01T00:00:00.000Z,2026-09-25T00:00:00.000Z,0
noaa_swpc_kp,1613,1613,2026-03-01T00:00:00.000Z,2026-09-25T12:00:00.000Z,0
